# CSE 579 Homework 2: Reinforcement Learning

In this assignment, you will implement:
1. **Policy Gradient (PG)** with a baseline
2. **Actor-Critic (AC)** with a single Q-function
3. **Soft Actor-Critic (SAC)** with double Q + entropy temperature

You will train and evaluate on **InvertedPendulum-v4**, and additionally run AC + SAC on **Ant-v4**.


**Getting started:** Click **File > Save a copy in Drive** to create your own editable copy of this notebook. Do not edit the original.


## 1. Setup and Installation


In [ ]:
# Clone the homework repository
!git clone https://github.com/WEIRDLabUW/CSE579-hw2.git
!cp -r CSE579-hw2/* .
!cp -r CSE579-hw2/.gitignore .


In [ ]:
# Install system dependencies (for MuJoCo rendering)
!apt-get install -y \
    libgl1-mesa-dev \
    libgl1-mesa-glx \
    libglew-dev \
    libosmesa6-dev \
    software-properties-common \
    patchelf

# Install PyTorch with CUDA support
!pip install -q \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --extra-index-url https://download.pytorch.org/whl/cu124

# Install gymnasium + mujoco + imageio for video rendering.
# (We do not need gymnasium-robotics — it pins mujoco<3.0, which lacks prebuilt
# wheels for Colab's Python and forces a source build that fails.)
!pip install -q \
    gymnasium==0.29.1 \
    mujoco \
    "imageio[ffmpeg]" \
    matplotlib \
    "numpy>=1.24,<2.0" \
    tqdm

import os
os.environ['LD_PRELOAD'] = ':/usr/lib/x86_64-linux-gnu/libGLEW.so'

# Restart the runtime so the new numpy version takes effect.
# After restart, skip this cell and continue from the next one.
os.kill(os.getpid(), 9)


**Important:** The cell above restarts the runtime. After it restarts, skip cells 1 & 2 above and continue from the cell below.


In [ ]:
# Re-set env var (lost after runtime restart)
import os
os.environ['LD_PRELOAD'] = ':/usr/lib/x86_64-linux-gnu/libGLEW.so'
os.environ['MUJOCO_GL'] = 'egl'


## 2. Imports and Device Setup


In [ ]:
import random

import gymnasium as gym
import numpy as np
import torch
import torch.nn.functional as F
from torch import optim

from agents import GenericACAgent, train_agent
from networks import PGPolicy, PGBaseline
from rollouts import evaluate, evaluate_agent, rollout
from utils import ReplayBuffer, log_density

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('using device:', device)


## 3. Policy Gradient (PG)

Fill in the TODOs in `train_model` below. You'll need to compute the return-to-go, then train the baseline by regression and the policy via the policy-gradient surrogate.


In [ ]:
def train_model(policy, baseline, trajs, policy_optim, baseline_optim, device, gamma=0.99, baseline_train_batch_size=64,
                baseline_num_epochs=5):
    states_all = []
    actions_all = []
    returns_all = []
    for traj in trajs:
        states_singletraj = traj['observations']
        actions_singletraj = traj['actions']
        rewards_singletraj = traj['rewards']
        returns_singletraj = np.zeros_like(rewards_singletraj)
        #========== TODO: start ==========
        # Compute the return to go on the current batch of trajectories
        # Hint: Go through all the trajectories in trajs and compute their return to go: discounted sum of rewards from that timestep to the end.
        # Hint: This is easy to do if you go backwards in time and sum up the reward as a running sum.
        # Hint: Remember that return to go is return = r[t] + gamma*r[t+1] + gamma^2*r[t+2] + .... Don't forget the discount!



        # returns_singletraj = ...
        #========== TODO: END ==========
        states_all.append(states_singletraj)
        actions_all.append(actions_singletraj)
        returns_all.append(returns_singletraj)
    states = np.concatenate(states_all)
    actions = np.concatenate(actions_all)
    returns = np.concatenate(returns_all)

    # TODO: Normalize the returns by subtracting mean and dividing by std
    # Hint: (return - return.mean()) / (return.std() + EPS), where EPS is a small constant for numerics
    # TODO start
    # returns = ...
    # TODO end

    criterion = torch.nn.MSELoss()
    n = len(states)
    arr = np.arange(n)
    for epoch in range(baseline_num_epochs):
        np.random.shuffle(arr)
        for i in range(n // baseline_train_batch_size):
            batch_index = arr[baseline_train_batch_size * i: baseline_train_batch_size * (i + 1)]
            batch_index = torch.LongTensor(batch_index).to(device)
            inputs = torch.Tensor(states).to(device)[batch_index]
            target = torch.Tensor(returns).to(device)[batch_index]
            
            #========== TODO: start ==========
            # Train baseline by regressing onto returns.
            # Hint: Regress the baseline from each state onto the above
            # computed return to go. You can use similar code to behavior cloning to do so. This should be
            # 2 lines of code
            # Hint: baseline is a callable function
            



            #========== TODO: END ==========
            baseline_optim.zero_grad()
            loss.backward()
            baseline_optim.step()

    action, std, logstd = policy(torch.Tensor(states).to(device))
    log_policy = log_density(torch.Tensor(actions).to(device), policy.mu, std, logstd)
    baseline_pred = baseline(torch.from_numpy(states).float().to(device))
    #========== TODO: start ==========
    # Train policy by optimizing surrogate objective: -log prob * (return - baseline)
    # Hint: log_policy and baseline_pred are already computed above; use them directly.
    # Hint: Policy gradient is given by: \grad log prob(a|s)* (return - baseline)
    # Hint: Then simply compute the surrogate objective by taking the objective as -log prob * (return - baseline)
    # Hint: You can then use standard pytorch machinery to take *one* gradient step on the policy


    
    #========== TODO: END ==========
    policy_optim.zero_grad()
    loss.backward()
    policy_optim.step()

    del states, actions, returns, states_all, actions_all, returns_all


# Training loop for policy gradient
def simulate_policy_pg(env, policy, baseline, num_epochs=200, batch_size=100,
                       gamma=0.99, baseline_train_batch_size=64, baseline_num_epochs=5, print_freq=10, device = "cuda", render=False):
    policy_optim = optim.Adam(policy.parameters())
    baseline_optim = optim.Adam(baseline.parameters())

    for iter_num in range(num_epochs):
        sample_trajs = []

        # Sampling trajectories
        for it in range(batch_size):
            sample_traj = rollout(
                env,
                policy,
                render=False)
            sample_trajs.append(sample_traj)

        # Logging returns occasionally
        if iter_num % print_freq == 0:
            rewards_np = np.mean(np.asarray([traj['rewards'].sum() for traj in sample_trajs]))
            path_length = np.max(np.asarray([traj['rewards'].shape[0] for traj in sample_trajs]))
            print("Episode: {}, reward: {}, max path length: {}".format(iter_num, rewards_np, path_length))

        # Training model
        train_model(policy, baseline, sample_trajs, policy_optim, baseline_optim, device, gamma=gamma,
                    baseline_train_batch_size=baseline_train_batch_size, baseline_num_epochs=baseline_num_epochs)


## 4. Actor-Critic (AC)

Fill in the TODOs in `ActorCriticAgent` below.


In [ ]:
class ActorCriticAgent(GenericACAgent):
    def update_actor(self, obs):
        dist = self.actor(obs)
        # The .rsample() is used to sample using the reparameterization trick to allow for backpropagation
        action = dist.rsample()
        log_prob = dist.log_prob(action).sum(-1, keepdim=True)
        #========== TODO: start ==========
        # Implement the actor update
        # Compute the Q values of the action using self.critic(obs, action). In this case it is a single instead of
        # double Q function so you do not need to take a minimum.
        # The policy loss is the mean over the negative Q values i.e we want to maximize the Q values

        
        
        #========== TODO: end ==========
        # optimize the actor
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        return actor_loss.item(), 0, 0

    def update_critic(self, obs, action, reward, next_obs, not_done_no_max):
        #========== TODO: start ==========
        # Train the single Q function:
        # Hint: Step 1: Compute current Q predictions using the obs and action and self.critic()
        # Hint: Step 2: Compute q targets using reward + critic_target * not_done_no_max for next_obs and
        # next actions sampled from the current policy. Use torch.no_grad() for this step to disable
        # gradient flow to the critic_target and the actor.
        # Hint: Step 3: Compute Bellman error as mean squared error between q_predictions and q_targets
       


        #========== TODO: end ==========
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        return critic_loss.item()

## 5. Soft Actor-Critic (SAC)

Fill in the TODOs in `SACAgent` below.


In [ ]:
class SACAgent(GenericACAgent):
    def update_actor(self, obs):        
        #========== TODO: start ==========
        # Sample actions and the log_prob of the actions from the actor given obs. Hint: This is the same as AC agent.
        # Get the two Q values from the double Q function critic and take the minimum value. Then calculate the actor loss which
        # is defined by self.alpha * log_prob - actor_Q. Make sure that gradient does not flow through the alpha parameter.

        
        #========== TODO: end ==========
        # optimize the actor
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # Update the temperature parameter to achieve entropy close to the target entropy
        self.log_alpha_optimizer.zero_grad()
        alpha_loss = (self.alpha *
                      (-log_prob - self.target_entropy).detach())
        alpha_loss = alpha_loss.mean()
        alpha_loss.backward()
        self.log_alpha_optimizer.step()
        return actor_loss.item(), -log_prob.mean().item(), alpha_loss.item()
    
    def update_critic(self, obs, action, reward, next_obs, not_done_no_max):
        #========== TODO: start ==========
        # Train the double Q function:
        # Hint step 1: Sample the next_action and log_prob of the next action using the self.actor and the next_obs. Use the code 
        # below in update_actor as a reference on how to do this
        
        # Hint step 2: Sample the two target Q values from the critic_target using next_obs and the sampled next_action. 
        # Calculate the target value by taking the min of the values and then subtracting self.alpha * log_prob
        # The target Q is the reward + (not_done_no_max * discount * target_value)
        # Hint: make sure gradients don't flow through the target_Q

        
        # Hint step 3:
        # Sample the current Q1 and Q2 values of the current state using the critic, and regress onto the target Q.
        # The loss is mse(Q1, targetQ) + mse(Q2 + target Q)

        
        #========== TODO: end ==========
        # Optimize the critic
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        return critic_loss.item()

---


## 6. Train and Evaluate


In [ ]:
class Args:
    def __init__(self, task, env, test=False, render=False, seed=0):
        # task: 'pg' (or 'policy_gradient'), 'actor_critic' (or 'ac'), 'sac'
        self.task = {'policy_gradient': 'pg', 'ac': 'actor_critic'}.get(task, task)
        self.env = env     # 'pendulum' or 'ant'
        self.test = test
        self.render = render
        self.seed = seed


def make_env(name, render=False):
    if name == 'pendulum':
        env = gym.make("InvertedPendulum-v4", render_mode='rgb_array' if render else None)
        max_episode_steps = 200
    elif name == 'ant':
        env = gym.make("Ant-v4", render_mode='rgb_array' if render else None)
        max_episode_steps = 500
    else:
        raise ValueError(f"unknown env: {name}")
    return gym.wrappers.TimeLimit(env, max_episode_steps=max_episode_steps)


def train_and_eval(args):
    """Train (or load) one task on one env with the given seed, then evaluate."""
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    random.seed(args.seed)
    np.random.seed(args.seed)

    env = make_env(args.env)
    env.reset(seed=args.seed)
    env.action_space.seed(args.seed)

    obs_size = env.observation_space.shape[0]
    ac_size = env.action_space.shape[0]
    action_range = [float(env.action_space.low.min()), float(env.action_space.high.max())]

    if args.task == 'pg':
        policy = PGPolicy(obs_size, ac_size, hidden_dim=64, hidden_depth=2).to(device)
        baseline = PGBaseline(obs_size, hidden_dim=64, hidden_depth=2).to(device)

        if not args.test:
            simulate_policy_pg(env, policy, baseline,
                               num_epochs=100, batch_size=64, gamma=0.99,
                               baseline_train_batch_size=64, baseline_num_epochs=5,
                               print_freq=10, device=device, render=False)
            torch.save(policy.state_dict(), f'pg_{args.env}_final.pth')
        else:
            policy.load_state_dict(torch.load(f'pg_{args.env}_final.pth'))
        evaluate(env, policy, num_validation_runs=100, render=False)
        return policy

    # AC / SAC
    num_train_steps = 30_000
    replay_buffer = ReplayBuffer(obs_size, ac_size, num_train_steps, device)
    if args.task == 'actor_critic':
        agent = ActorCriticAgent(
            obs_dim=obs_size, action_dim=ac_size, action_range=action_range, device=device,
            discount=0.99, actor_lr=3e-4, critic_lr=3e-4, critic_tau=5e-3, batch_size=256,
            hidden_dim=256, hidden_depth=2, double_critic=False, temperature=False,
        )
    elif args.task == 'sac':
        agent = SACAgent(
            obs_dim=obs_size, action_dim=ac_size, action_range=action_range, device=device,
            discount=0.99, init_temperature=0.1, alpha_lr=3e-4,
            actor_lr=3e-4, critic_lr=3e-4, critic_tau=0.005, batch_size=256,
            target_entropy=-ac_size, hidden_dim=256, hidden_depth=2,
            double_critic=True, temperature=True,
        )
    else:
        raise ValueError(f"unknown task: {args.task}")

    if not args.test:
        train_agent(agent, env, num_train_steps=num_train_steps, num_seed_steps=5000,
                    eval_frequency=10_000, num_eval_episodes=10, replay_buffer=replay_buffer)
        agent.save(f'{args.task}_{args.env}_final.pth')
    else:
        agent.load(f'{args.task}_{args.env}_final.pth')
    evaluate_agent(env, agent, "final", num_episodes=100, verbose=True)
    return agent


### 6a. PG — Pendulum


In [ ]:
args = Args('pg', 'pendulum', seed=1)
result = train_and_eval(args)


### 6b. Actor-Critic — Pendulum


In [ ]:
args = Args('actor_critic', 'pendulum')
result = train_and_eval(args)


### 6c. SAC — Pendulum


In [ ]:
args = Args('sac', 'pendulum')
result = train_and_eval(args)


### 6d. Actor-Critic — Ant


In [ ]:
args = Args('actor_critic', 'ant')
result = train_and_eval(args)


### 6e. SAC — Ant


In [ ]:
args = Args('sac', 'ant')
result = train_and_eval(args)


---
## Submission

Download the trained `.pth` weights below and submit them along with your filled-in `policy_gradient.py`, `actor_critic.py`, and `sac.py`.


In [ ]:
import glob

# If running on Google Colab, trigger download of weight files
try:
    from google.colab import files
    for f in glob.glob("*.pth"):
        files.download(f)
except ImportError:
    print("Not on Colab. Weights are saved in the current directory:")
    for f in glob.glob("*.pth"):
        print(" ", f)
